## Task 1: Import All the Needed Packages

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve


import joblib
import os
import seaborn as sns
from tqdm import tqdm

# Set consistent plotting style
sns.set(color_codes=True)
plt.rcParams["figure.figsize"] = (8,5)
print(pd.__version__)

## Task 2: Statistical Analysis and Data Visualization

### 2.1 Statistical Analysis

In [ ]:
# load data

df = pd.read_csv('../data/indian_liver_patient.csv')

print(f"\nDataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

print("\nFirst 5 rows:")
print(df.head())

In [ ]:
# identify column types

numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

print(f"Numerical columns ({len(numerical_cols)}): {numerical_cols}")
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")

In [ ]:
# descriptive statistics

print("\nDescriptive statistics:")
print(df.describe())

# calculate stats for each numerical column
stats_list = []
for col in numerical_cols:
    data = df[col]
    stats_list.append({
        'Feature': col,
        'Count': data.count(),
        'Mean': data.mean(),
        'Median': data.median(),
        'Mode': data.mode()[0] if len(data.mode()) > 0 else np.nan,
        'Std': data.std(),
        'Variance': data.var(),
        'Min': data.min(),
        'Max': data.max(),
        'Range': data.max() - data.min(),
        'Q1': data.quantile(0.25),
        'Q3': data.quantile(0.75),
        'IQR': data.quantile(0.75) - data.quantile(0.25),
        'Skewness': data.skew(),
        'Kurtosis': data.kurtosis()
    })

stats_df = pd.DataFrame(stats_list)
print("\nStatistical measures:")
print(stats_df.to_string(index=False))

In [ ]:
# correlation analysis

corr_matrix = df[numerical_cols].corr()
print("\nCorrelation matrix:")
print(corr_matrix.round(3))

# correlations with target variable
target = 'Dataset'
target_corr = corr_matrix[target].sort_values(ascending=False)

print("\nCorrelations with target variable:")
for feat in target_corr.index:
    if feat != target:
        print(f"  {feat}: {target_corr[feat]:.4f}")

# highly correlated feature pairs
print("\nHighly correlated feature pairs (|r| > 0.7):")
found = False

for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        col1 = corr_matrix.columns[i]
        col2 = corr_matrix.columns[j]
        r = corr_matrix.iloc[i, j]
        
        # exclude target variable and check threshold
        if abs(r) > 0.7 and col1 != target and col2 != target:
            print(f"  {col1} - {col2}: {r:.4f}")
            found = True

if not found:
    print("  None found")

In [ ]:
# target variable distribution
target_counts = df[target].value_counts().sort_index()

print("\nTarget variable distribution:")
for val, count in target_counts.items():
    pct = (count/len(df))*100
    print(f"  Class {val}: {count} ({pct:.1f}%)")

# calculate imbalance ratio
ratio = target_counts.max() / target_counts.min()
print(f"Imbalance ratio: {ratio:.2f}:1")

In [ ]:
# normality tests
# shapiro-wilk test for each feature
normality = []
for col in numerical_cols:
    if col != target:
        test_data = df[col].dropna()
        # shapiro test requires 3-5000 samples
        if 3 <= len(test_data) <= 5000:
            w, p = stats.shapiro(test_data)
            normality.append({
                'Feature': col,
                'W': w,
                'p-value': p,
                'Normal': 'Yes' if p > 0.05 else 'No'
            })

norm_df = pd.DataFrame(normality)
print("\nNormality tests (Shapiro-Wilk):")
print(norm_df.to_string(index=False))

In [ ]:
# distribution characteristics
print("\nDistribution characteristics:")
for _, row in stats_df.iterrows():
    print(f"\n{row['Feature']}:")
    
    # interpret skewness
    if abs(row['Skewness']) < 0.5:
        skew = "symmetric"
    elif row['Skewness'] > 0.5:
        skew = "right-skewed"
    else:
        skew = "left-skewed"
    
    # interpret kurtosis
    if abs(row['Kurtosis']) < 0.5:
        kurt = "mesokurtic (normal)"
    elif row['Kurtosis'] > 0.5:
        kurt = "leptokurtic (peaked)"
    else:
        kurt = "platykurtic (flat)"
    
    print(f"  Skewness: {row['Skewness']:.3f} ({skew})")
    print(f"  Kurtosis: {row['Kurtosis']:.3f} ({kurt})")


In [ ]:
# save dataframes to csv files
stats_df.to_csv('../outputs/data_analysis_and_visualization/statistical_summary.csv', index=False)
corr_matrix.to_csv('../outputs/data_analysis_and_visualization/correlation_matrix.csv')
norm_df.to_csv('../outputs/data_analysis_and_visualization/normality_tests.csv', index=False)

### 2.2 Data Visualization

In [ ]:
# Create 'plots' folder inside the 'outputs/data_analysis_and_visualization' directory
PLOTS_DIR = os.path.join("..", "outputs/data_analysis_and_visualization", "plots")

if not os.path.exists(PLOTS_DIR):
    os.makedirs(PLOTS_DIR)

print("Folder '../outputs/data_analysis_and_visualization/plots' ready.")


In [ ]:
# Generate histograms for all numeric columns
num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

for col in num_cols:
    # Histogram
    plt.figure()
    df[col].dropna().hist()
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, f"hist_{col}.png"))
    plt.close()

print("Histograms saved in 'plots/' folder.")


In [ ]:
# Generate Boxplots for all numeric columns
for col in num_cols:
# Boxplot
    plt.figure()
    df[col].dropna().plot(kind="box")
    plt.title(f"Boxplot of {col}")
    plt.ylabel(col)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, f"box_{col}.png"))
    plt.close()

print("Boxplots saved in 'plots/' folder.")


In [ ]:
# To find files in current folder or in outputs/
def find_file_possibly_in_outputs(filename):
    possible_paths = [
        filename,
        f"../outputs/data_analysis_and_visualization/{filename}",
        f"./outputs/data_analysis_and_visualization/{filename}",
    ]
    for p in possible_paths:
        if os.path.exists(p):
            return p
    return None


In [ ]:
# Correlation Heatmap
corr_path = find_file_possibly_in_outputs("correlation_matrix.csv")

if corr_path:
    corr_df = pd.read_csv(corr_path)
    first = corr_df.columns[0].lower()
    if first in ["unnamed: 0", "feature", "index"]:
        corr_df = corr_df.set_index(corr_df.columns[0])
    corr_df = corr_df.apply(pd.to_numeric, errors="coerce")
else:
    # fallback: compute from df
    corr_df = df.select_dtypes(include=["int64", "float64"]).corr()

plt.figure(figsize=(6,5))
plt.imshow(corr_df.values.astype(float), aspect="auto")
plt.colorbar()
plt.xticks(range(len(corr_df.columns)), corr_df.columns, rotation=90)
plt.yticks(range(len(corr_df.index)), corr_df.index)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "correlation_heatmap.png"))
plt.close()

print("Correlation heatmap saved in 'plots/' folder.")
if corr_path:
    print(f"(Loaded from: {corr_path})")
else:
    print("(Computed directly from dataset.)")


In [ ]:
# Normality Test Visualization
normality_path = find_file_possibly_in_outputs("normality_tests.csv")

if normality_path:
    normality_df = pd.read_csv(normality_path)
    # your file uses 'Feature' and 'p-value'
    normality_df.columns = [c.lower() for c in normality_df.columns]

    if "feature" in normality_df.columns and "p-value" in normality_df.columns:
        plt.figure(figsize=(8,4))
        plt.bar(normality_df["feature"], normality_df["p-value"])
        plt.xticks(rotation=45, ha="right")
        plt.ylabel("p-value")
        plt.title("Normality Test p-values")
        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR, "normality_pvalues.png"))
        plt.close()
        print(f"Normality chart saved. (Loaded from: {normality_path})")
    else:
        print("File found but columns are not 'Feature' and 'p-value'.")
else:
    print("normality_tests.csv not found in expected folders.")


## Task 3: Data preprocessing

### 3.1 Convert target variable to binary

In [ ]:
# 1 = Liver Disease, 0 = No Liver Disease
df['Dataset'] = df['Dataset'].map({1: 1, 2: 0})  # Keep 1 as 1, change 2 to 0

In [ ]:
# Verify the conversion
print(f"Target distribution after binary encoding:")
print(f"  Class 1 (Disease): {(df['Dataset'] == 1).sum()} ({(df['Dataset'] == 1).mean()*100:.1f}%)")
print(f"  Class 0 (No Disease): {(df['Dataset'] == 0).sum()} ({(df['Dataset'] == 0).mean()*100:.1f}%)")

### 3.2 Encoding Categorical Variables

The column Gender contains Male / Female.

In [ ]:
df['Gender'] = df['Gender'].map({'Male':1, 'Female':0})
df['Gender'].unique()

### 3.3 Splitting the data

In [ ]:
# Split data FIRST to prevent data leakage
print("Splitting data into training and test sets...")
X = df.drop('Dataset', axis=1)
y = df['Dataset']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Training target distribution: {y_train.value_counts().to_dict()}")
print(f"Test target distribution: {y_test.value_counts().to_dict()}")

### 3.4 Handle Missing Values

In [ ]:
# Check missing values
df.isnull().sum()

imputation_col = 'Albumin_and_Globulin_Ratio'

# Check for missing values in the training set
train_missing_count = X_train[imputation_col].isnull().sum()
test_missing_count = X_test[imputation_col].isnull().sum()

print(f"\nHandling Missing Values in '{imputation_col}':")
print(f"  Training set missing: {train_missing_count}")
print(f"  Test set missing: {test_missing_count}")

# Calculate the median ONLY from the training data (CRITICAL for no leakage)
train_median = X_train[imputation_col].median()
print(f"  Median calculated from training data: {train_median:.4f}")

# Impute missing values in BOTH X_train and X_test using the train_median
X_train = X_train.copy()
X_test = X_test.copy()
X_train[imputation_col] = X_train[imputation_col].fillna(train_median)
X_test[imputation_col] = X_test[imputation_col].fillna(train_median)

# Verification
print("\nVerification of Missing Value Imputation:")
print(f"  Missing values in X_train: {X_train[imputation_col].isnull().sum()}")
print(f"  Missing values in X_test: {X_test[imputation_col].isnull().sum()}")

## Task 4: Outlier Analysis

### 4.1 Compute IQR outlier summary

In [ ]:
# Define continuous features only (exclude binary and target variables)
continuous_features = ['Age', 'Total_Bilirubin', 'Direct_Bilirubin', 
                      'Alkaline_Phosphotase', 'Alamine_Aminotransferase', 
                      'Aspartate_Aminotransferase', 'Total_Protiens', 
                      'Albumin', 'Albumin_and_Globulin_Ratio']

print(f"Analyzing outliers for {len(continuous_features)} continuous features:")
print(continuous_features)

# Compute IQR bounds and outlier counts for CONTINUOUS features only
out_rows = []

for col in tqdm(continuous_features):
    s = pd.to_numeric(X_train[col], errors="coerce")
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    mask_out = (s < lower) | (s > upper)
    non_null_n = s.notna().sum()
    out_count = mask_out.sum()
    out_pct = (out_count / non_null_n * 100.0) if non_null_n > 0 else 0.0

    out_rows.append({
        "feature": col,
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "lower_bound": lower,
        "upper_bound": upper,
        "outliers": int(out_count),
        "non_null_n": int(non_null_n),
        "outlier_pct": out_pct
    })

outlier_summary = pd.DataFrame(out_rows).sort_values("outlier_pct", ascending=False)
print("\nOutlier Summary (Continuous Features Only):")
print(outlier_summary.head(10))


### 4.2 Visualize distributions

In [ ]:
plots_dir = "../outputs/outlier_outputs/outlier_plots"
os.makedirs(plots_dir, exist_ok=True)

# Plot histograms and boxplots for top outlier-prone continuous features
k = min(6, len(outlier_summary))
cols_of_interest = outlier_summary.head(k)["feature"].tolist()

print(f"\nCreating plots for top {k} outlier-prone features:")
print(cols_of_interest)

for c in cols_of_interest:
    s = pd.to_numeric(X_train[c], errors="coerce").dropna()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    # Histogram with IQR bounds
    plt.figure(figsize=(10, 6))
    sns.histplot(s, kde=True, bins=40)
    plt.axvline(lower, color="red", linestyle="--", label="IQR lower", alpha=0.7)
    plt.axvline(upper, color="red", linestyle="--", label="IQR upper", alpha=0.7)
    plt.axvline(q1, color="orange", linestyle=":", label="Q1", alpha=0.7)
    plt.axvline(q3, color="orange", linestyle=":", label="Q3", alpha=0.7)
    plt.title(f"{c} - Distribution with IQR Bounds\n({outlier_summary[outlier_summary['feature']==c]['outlier_pct'].values[0]:.1f}% outliers)")
    plt.xlabel(c)
    plt.ylabel("Frequency")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, f"{c}_histogram.png"), dpi=150, bbox_inches="tight")
    plt.close()

    # Boxplot with IQR bounds
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=s, orient="h")
    plt.axvline(lower, color="red", linestyle="--", label="IQR lower", alpha=0.7)
    plt.axvline(upper, color="red", linestyle="--", label="IQR upper", alpha=0.7)
    plt.title(f"{c} - Boxplot with IQR Bounds")
    plt.xlabel(c)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, f"{c}_boxplot.png"), dpi=150, bbox_inches="tight")
    plt.close()

print(f"Visualizations saved to '{plots_dir}' directory")

### 4.3 Outlier Treatment Create Winsorized

In [ ]:
def winsorize_features(X_train, X_test, continuous_features):
    """
    Apply winsorization using IQR bounds computed from training data only
    to prevent data leakage
    """
    winsorized_train = X_train.copy()
    winsorized_test = X_test.copy()
    
    for col in tqdm(continuous_features, desc="Winsorizing features"):
        # Compute IQR bounds ONLY on training data
        s_train = pd.to_numeric(X_train[col], errors="coerce")
        q1 = s_train.quantile(0.25)
        q3 = s_train.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        # Apply same bounds to both train and test sets
        winsorized_train[col] = s_train.clip(lower=lower, upper=upper)
        winsorized_test[col] = pd.to_numeric(X_test[col], errors="coerce").clip(lower=lower, upper=upper)
    
    return winsorized_train, winsorized_test

# Apply winsorization using training data statistics
print("\nApplying winsorization to prevent data leakage...")
X_train_wins, X_test_wins = winsorize_features(X_train, X_test, continuous_features)

print("\nFirst 5 rows of winsorized training dataset:")
print(X_train_wins.head())

### 4.4 Compare Original vs Winsorized Statistics

In [ ]:
# Compare summary stats for continuous features only (TRAINING DATA ONLY)
print("\nComparing Original vs Winsorized Statistics...")

summ_rows = []
for col in continuous_features:
    s0 = pd.to_numeric(X_train[col], errors="coerce")  # original training data
    s1 = pd.to_numeric(X_train_wins[col], errors="coerce")  # winsorized training data

    # Calculate outlier counts using training data bounds
    q1 = s0.quantile(0.25)
    q3 = s0.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    
    outliers_removed = ((s0 < lower) | (s0 > upper)).sum()

    summ_rows.append({
        "feature": col,
        "orig_mean": s0.mean(),
        "winsor_mean": s1.mean(),
        "mean_change_pct": ((s1.mean() - s0.mean()) / s0.mean()) * 100,
        "orig_std": s0.std(),
        "winsor_std": s1.std(),
        "std_change_pct": ((s1.std() - s0.std()) / s0.std()) * 100,
        "orig_min": s0.min(),
        "winsor_min": s1.min(),
        "orig_max": s0.max(),
        "winsor_max": s1.max(),
        "outliers_removed": outliers_removed,
        "outlier_pct": (outliers_removed / len(s0)) * 100,
        "lower_bound": lower,
        "upper_bound": upper
    })

wins_compare = pd.DataFrame(summ_rows).sort_values("outliers_removed", ascending=False)
print("\nStatistical Comparison - Training Data Only (Original vs Winsorized):")
print(wins_compare.round(4))

# Also show test set statistics for comparison
test_summ_rows = []
for col in continuous_features:
    s0_test = pd.to_numeric(X_test[col], errors="coerce")  # original test data
    s1_test = pd.to_numeric(X_test_wins[col], errors="coerce")  # winsorized test data
    
    test_summ_rows.append({
        "feature": col,
        "test_orig_mean": s0_test.mean(),
        "test_winsor_mean": s1_test.mean(),
        "test_mean_change_pct": ((s1_test.mean() - s0_test.mean()) / s0_test.mean()) * 100 if s0_test.mean() != 0 else 0,
    })

test_compare = pd.DataFrame(test_summ_rows)
print("\nTest Set Impact - Mean Changes:")
print(test_compare[['feature', 'test_mean_change_pct']].round(4))

# Create visualization of the impact
plt.figure(figsize=(12, 8))

# Plot 1: Mean changes
plt.subplot(2, 2, 1)
changes = wins_compare.set_index('feature')['mean_change_pct']
colors = ['red' if x < 0 else 'green' for x in changes]
plt.barh(changes.index, changes.values, color=colors)
plt.axvline(0, color='black', linestyle='-', alpha=0.3)
plt.title('Mean Change % After Winsorization\n(Training Data)')
plt.xlabel('Percentage Change (%)')
plt.grid(axis='x', alpha=0.3)

# Plot 2: Standard deviation changes
plt.subplot(2, 2, 2)
std_changes = wins_compare.set_index('feature')['std_change_pct']
colors_std = ['red' if x < 0 else 'green' for x in std_changes]
plt.barh(std_changes.index, std_changes.values, color=colors_std)
plt.axvline(0, color='black', linestyle='-', alpha=0.3)
plt.title('Std Dev Change % After Winsorization\n(Training Data)')
plt.xlabel('Percentage Change (%)')
plt.grid(axis='x', alpha=0.3)

# Plot 3: Outliers removed
plt.subplot(2, 2, 3)
outliers_removed = wins_compare.set_index('feature')['outliers_removed']
plt.barh(outliers_removed.index, outliers_removed.values, color='orange')
plt.title('Number of Outliers Removed\n(Training Data)')
plt.xlabel('Number of Outliers')
plt.grid(axis='x', alpha=0.3)

# Plot 4: Outlier percentage
plt.subplot(2, 2, 4)
outlier_pct = wins_compare.set_index('feature')['outlier_pct']
plt.barh(outlier_pct.index, outlier_pct.values, color='purple')
plt.title('Outlier Percentage %\n(Training Data)')
plt.xlabel('Percentage of Data (%)')
plt.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/outlier_outputs/winsorization_impact.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nKey Insights from Winsorization:")
print(f"- Features with most outliers removed: {wins_compare.iloc[0]['feature']} ({wins_compare.iloc[0]['outliers_removed']} outliers)")
print(f"- Maximum mean reduction: {wins_compare['mean_change_pct'].min():.1f}%")
print(f"- Maximum std reduction: {wins_compare['std_change_pct'].min():.1f}%")

### 4.5 Save outputs

In [ ]:
# Create output directory
out_dir = "../outputs/outlier_outputs"
os.makedirs(out_dir, exist_ok=True)

# Create complete winsorized datasets by combining features with target
winsorized_train_complete = pd.concat([X_train_wins, y_train.reset_index(drop=True)], axis=1)
winsorized_test_complete = pd.concat([X_test_wins, y_test.reset_index(drop=True)], axis=1)

# Define file paths
outlier_summary_path = os.path.join(out_dir, "iqr_outlier_summary.csv")
winsorized_train_path = os.path.join(out_dir, "winsorized_train_dataset.csv")
winsorized_test_path = os.path.join(out_dir, "winsorized_test_dataset.csv")
wins_compare_path = os.path.join(out_dir, "winsor_vs_original_comparison.csv")
test_compare_path = os.path.join(out_dir, "test_set_winsorization_impact.csv")

# Save files
outlier_summary.to_csv(outlier_summary_path, index=False)
winsorized_train_complete.to_csv(winsorized_train_path, index=False)
winsorized_test_complete.to_csv(winsorized_test_path, index=False)
wins_compare.to_csv(wins_compare_path, index=False)
test_compare.to_csv(test_compare_path, index=False)
print("OUTLIER ANALYSIS OUTPUT FILES SAVED IN DIRECTORY: " + out_dir)

# Print summary statistics
print(f"\nSUMMARY STATISTICS:")
print(f"- Total outliers removed from training data: {wins_compare['outliers_removed'].sum()}")
print(f"- Features with >10% outliers: {len(wins_compare[wins_compare['outlier_pct'] > 10])}")
print(f"- Most affected feature: {wins_compare.iloc[0]['feature']} ({wins_compare.iloc[0]['outlier_pct']:.1f}% outliers)")

# Save the IQR bounds for future reference
iqr_bounds = []
for col in continuous_features:
    s_train = pd.to_numeric(X_train[col], errors="coerce")
    q1 = s_train.quantile(0.25)
    q3 = s_train.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    
    iqr_bounds.append({
        'feature': col,
        'q1': q1,
        'q3': q3,
        'iqr': iqr,
        'lower_bound': lower,
        'upper_bound': upper
    })

iqr_bounds_df = pd.DataFrame(iqr_bounds)
iqr_bounds_path = os.path.join(out_dir, "iqr_bounds_training.csv")
iqr_bounds_df.to_csv(iqr_bounds_path, index=False)
print(f"IQR bounds saved: {iqr_bounds_path}")

print("\nOutlier analysis completed successfully!")

### 3.5 Feature Scaling
Applying feature scaling after outlier treatment (using training data statistics only)

In [ ]:
print("Applying feature scaling using training data statistics only...")

# First, ensure output directory exists
os.makedirs('../outputs/preprocessed_data', exist_ok=True)

# Select only numerical columns for scaling from winsorized data
features_to_scale_train = X_train_wins[continuous_features]
other_features_train = X_train_wins.drop(columns=continuous_features)  # Gender

features_to_scale_test = X_test_wins[continuous_features]
other_features_test = X_test_wins.drop(columns=continuous_features)  # Gender

print(f"Features to scale: {list(features_to_scale_train.columns)}")
print(f"Other features: {list(other_features_train.columns)}")

# Standard Scaling - Fit on training, transform both train and test
scaler = StandardScaler()

print("\nFitting scaler on training data and transforming both sets...")

# Fit scaler ONLY on training data, then transform both sets
scaled_array_train = scaler.fit_transform(features_to_scale_train)
scaled_array_test = scaler.transform(features_to_scale_test)

# Create final scaled DataFrames
scaled_features_train_df = pd.DataFrame(
    scaled_array_train, 
    columns=continuous_features,
    index=features_to_scale_train.index  # Preserve original indices
)

scaled_features_test_df = pd.DataFrame(
    scaled_array_test, 
    columns=continuous_features,
    index=features_to_scale_test.index  # Preserve original indices
)

print("Feature scaling completed successfully!")

# Reconstruct complete datasets
print("\nReconstructing final datasets...")
final_train_df = pd.concat([
    scaled_features_train_df, 
    other_features_train,  # Already has correct index
    y_train
], axis=1)

final_test_df = pd.concat([
    scaled_features_test_df, 
    other_features_test,  # Already has correct index
    y_test
], axis=1)

print("Training set shape after preprocessing:", final_train_df.shape)
print("Test set shape after preprocessing:", final_test_df.shape)

# Verify scaling worked correctly on training data
print("\nSCALING VALIDATION - TRAINING DATA")
print("-" * 50)
print("Training set statistics after scaling (should be ~mean=0, std=1 for continuous features):")
scaling_validation = scaled_features_train_df.describe().round(3)
print(scaling_validation.loc[['mean', 'std']])

# Verify target variable distribution is preserved
print("TARGET DISTRIBUTION VALIDATION")
print("-" * 50)
print("Training target distribution:")
train_target_counts = final_train_df['Dataset'].value_counts().sort_index()
for val, count in train_target_counts.items():
    pct = (count/len(final_train_df))*100
    status = "Disease" if val == 1 else "No Disease"
    print(f"  Class {val} ({status}): {count} ({pct:.1f}%)")

print("\nTest target distribution:")
test_target_counts = final_test_df['Dataset'].value_counts().sort_index()
for val, count in test_target_counts.items():
    pct = (count/len(final_test_df))*100
    status = "Disease" if val == 1 else "No Disease"
    print(f"  Class {val} ({status}): {count} ({pct:.1f}%)")

# Save the preprocessed datasets
final_train_df.to_csv('../outputs/preprocessed_data/train_dataset_preprocessed.csv', index=False)
final_test_df.to_csv('../outputs/preprocessed_data/test_dataset_preprocessed.csv', index=False)
print("Preprocessed datasets saved successfully!")

# Final validation summary
print(f"Final Dataset Summary:")
print(f"   - Training set: {final_train_df.shape[0]} samples, {final_train_df.shape[1]} features")
print(f"   - Test set: {final_test_df.shape[0]} samples, {final_test_df.shape[1]} features")
print(f"   - Continuous features scaled: {len(continuous_features)}")
print(f"   - Categorical features preserved: 1 (Gender)")
print(f"   - Target variable: Binary (1 = Disease, 0 = No Disease)")
print(f"Output files saved in: ../outputs/preprocessed_data/")


## Task 5: Model Implementation & Tuning 

In [ ]:
# After all preprocessing, prepare data for modeling
print("\nPreparing data for modeling...")
X_train_final = final_train_df.drop('Dataset', axis=1)
y_train_final = final_train_df['Dataset']
X_test_final = final_test_df.drop('Dataset', axis=1) 
y_test_final = final_test_df['Dataset']

print(f"\nFinal data shapes for modeling:")
print(f"X_train: {X_train_final.shape}, y_train: {y_train_final.shape}")
print(f"X_test: {X_test_final.shape}, y_test: {y_test_final.shape}")
print(f"Target distribution - Train: {y_train_final.value_counts().to_dict()}")
print(f"Target distribution - Test: {y_test_final.value_counts().to_dict()}")

# Ensure output directories exist
os.makedirs('../outputs/logistic_regression', exist_ok=True)
os.makedirs('../outputs/decision_tree', exist_ok=True)
os.makedirs('../outputs/random_forest', exist_ok=True)
os.makedirs('../outputs/model_comparison', exist_ok=True)

# Save the scaler for future use
joblib.dump(scaler, '../outputs/preprocessed_data/standard_scaler.pkl')
print("Scaler saved for future use!")

### 5.1 Logistic Regression implementation & hyperparameter tuning

#### 5.1.1 Baseline Model

In [ ]:
# Define baseline model
baseline_lr = LogisticRegression(random_state=42, max_iter=1000)

print("Performing 5-fold cross-validation for baseline Logistic Regression...")
cv_scores_lr = cross_val_score(baseline_lr, X_train_final, y_train_final, cv=5, scoring='f1')
print(f"Baseline LR Cross-Validation F1 Scores: {cv_scores_lr}")
print(f"Baseline LR CV F1 Mean: {cv_scores_lr.mean():.4f} (+/- {cv_scores_lr.std() * 2:.4f})")

# Additional cross-validation with different metrics
print("\nAdditional cross-validation metrics:")
cv_accuracy_lr = cross_val_score(baseline_lr, X_train_final, y_train_final, cv=5, scoring='accuracy')
cv_precision_lr = cross_val_score(baseline_lr, X_train_final, y_train_final, cv=5, scoring='precision')
cv_recall_lr = cross_val_score(baseline_lr, X_train_final, y_train_final, cv=5, scoring='recall')
cv_auc_lr = cross_val_score(baseline_lr, X_train_final, y_train_final, cv=5, scoring='roc_auc')

print(f"CV Accuracy:  {cv_accuracy_lr.mean():.4f} (+/- {cv_accuracy_lr.std() * 2:.4f})")
print(f"CV Precision: {cv_precision_lr.mean():.4f} (+/- {cv_precision_lr.std() * 2:.4f})")
print(f"CV Recall:    {cv_recall_lr.mean():.4f} (+/- {cv_recall_lr.std() * 2:.4f})")
print(f"CV AUC-ROC:   {cv_auc_lr.mean():.4f} (+/- {cv_auc_lr.std() * 2:.4f})")

# Now fit the baseline model on full training data
print("\nTraining baseline Logistic Regression on full training set...")
baseline_lr.fit(X_train_final, y_train_final)

# Make predictions
y_pred_base = baseline_lr.predict(X_test_final)
y_proba_base = baseline_lr.predict_proba(X_test_final)[:, 1]

# Calculate performance metrics
baseline_accuracy = accuracy_score(y_test_final, y_pred_base)
baseline_precision = precision_score(y_test_final, y_pred_base)
baseline_recall = recall_score(y_test_final, y_pred_base)
baseline_f1 = f1_score(y_test_final, y_pred_base)
baseline_auc = roc_auc_score(y_test_final, y_proba_base)

# Confusion matrix
cm_base = confusion_matrix(y_test_final, y_pred_base)

print("\nBaseline Logistic Regression Performance:")
print(f"Accuracy:  {baseline_accuracy:.4f}")
print(f"Precision: {baseline_precision:.4f}")
print(f"Recall:    {baseline_recall:.4f}")
print(f"F1-Score:  {baseline_f1:.4f}")
print(f"AUC-ROC:   {baseline_auc:.4f}")

print("\nConfusion Matrix:")
print(cm_base)
print(f"True Negatives: {cm_base[0,0]}, False Positives: {cm_base[0,1]}")
print(f"False Negatives: {cm_base[1,0]}, True Positives: {cm_base[1,1]}")

# Classification report
print("\nClassification Report:")
print(classification_report(y_test_final, y_pred_base, target_names=['No Disease', 'Disease']))

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm_base, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No Disease', 'Disease'],
            yticklabels=['No Disease', 'Disease'])
plt.title('Confusion Matrix - Baseline Logistic Regression')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('../outputs/logistic_regression/confusion_matrix_baseline.png', dpi=300, bbox_inches='tight')
plt.show()

# Plot ROC curve
fpr_base, tpr_base, _ = roc_curve(y_test_final, y_proba_base)
plt.figure(figsize=(8, 6))
plt.plot(fpr_base, tpr_base, label=f'Baseline LR (AUC = {baseline_auc:.3f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Baseline Logistic Regression')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/logistic_regression/roc_curve_baseline.png', dpi=300, bbox_inches='tight')
plt.show()

# Feature coefficients analysis
feature_importance_base = pd.DataFrame({
    'Feature': X_train_final.columns,
    'Coefficient': baseline_lr.coef_[0],
    'Absolute_Value': np.abs(baseline_lr.coef_[0])
}).sort_values('Absolute_Value', ascending=False)

print("\nTop 10 Most Important Features (Baseline Logistic Regression):")
print(feature_importance_base.head(10).round(4))

# Plot feature importance
plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance_base.head(10), 
            x='Coefficient', 
            y='Feature', 
            hue='Feature',  
            legend=False,    
            palette='coolwarm')
plt.axvline(0, color='black', linestyle='-', alpha=0.3)
plt.title('Top 10 Feature Coefficients - Baseline Logistic Regression')
plt.xlabel('Coefficient Value')
plt.tight_layout()
plt.savefig('../outputs/logistic_regression/feature_importance_baseline.png', dpi=300, bbox_inches='tight')
plt.show()

# Save baseline results
baseline_results = {
    'Model': 'Baseline_Logistic_Regression',
    'Accuracy': baseline_accuracy,
    'Precision': baseline_precision,
    'Recall': baseline_recall,
    'F1_Score': baseline_f1,
    'AUC_ROC': baseline_auc
}

baseline_performance_df = pd.DataFrame([baseline_results])
baseline_performance_df.to_csv('../outputs/logistic_regression/performance_baseline.csv', index=False)
feature_importance_base.to_csv('../outputs/logistic_regression/feature_importance_baseline.csv', index=False)

print("Baseline Logistic Regression completed and saved!")

#### 5.1.2 Hyperparameter Tuning

In [ ]:
# Hyperparameter Tuning with GridSearchCV
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear'],
    'class_weight': [None, 'balanced']
}

print("Starting GridSearchCV for Logistic Regression...")
grid_search = GridSearchCV(
    LogisticRegression(max_iter=2000, random_state=42), 
    param_grid, 
    cv=5, 
    scoring='f1',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train_final, y_train_final)

print(f"\nBest parameters found: {grid_search.best_params_}")
print(f"Best cross-validation score: {grid_search.best_score_:.4f}")


#### 5.1.3 Best Model

In [ ]:
best_lr = grid_search.best_estimator_
y_pred_best = best_lr.predict(X_test_final)
y_proba_best = best_lr.predict_proba(X_test_final)[:, 1]

#### 5.1.4 Results

In [ ]:
# Tuned model metrics
tuned_accuracy = accuracy_score(y_test_final, y_pred_best)
tuned_precision = precision_score(y_test_final, y_pred_best)
tuned_recall = recall_score(y_test_final, y_pred_best)
tuned_f1 = f1_score(y_test_final, y_pred_best)
tuned_auc = roc_auc_score(y_test_final, y_proba_best)

# Print comparison
print(f"Best Parameters: {grid_search.best_params_}")
print(f"\nPerformance Comparison:")
print(f"{'Metric':<12} {'Baseline':<10} {'Tuned':<10} {'Improvement':<12}")
print(f"{'-' * 45}")
print(f"{'Accuracy':<12} {baseline_accuracy:.4f}    {tuned_accuracy:.4f}    {tuned_accuracy - baseline_accuracy:+.4f}")
print(f"{'Precision':<12} {baseline_precision:.4f}    {tuned_precision:.4f}    {tuned_precision - baseline_precision:+.4f}")
print(f"{'Recall':<12} {baseline_recall:.4f}    {tuned_recall:.4f}    {tuned_recall - baseline_recall:+.4f}")
print(f"{'F1-Score':<12} {baseline_f1:.4f}    {tuned_f1:.4f}    {tuned_f1 - baseline_f1:+.4f}")
print(f"{'AUC-ROC':<12} {baseline_auc:.4f}    {tuned_auc:.4f}    {tuned_auc - baseline_auc:+.4f}")

# Plot comparison of baseline vs tuned
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
baseline_scores = [baseline_accuracy, baseline_precision, baseline_recall, baseline_f1, baseline_auc]
tuned_scores = [tuned_accuracy, tuned_precision, tuned_recall, tuned_f1, tuned_auc]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, baseline_scores, width, label='Baseline', alpha=0.7)
bars2 = ax.bar(x + width/2, tuned_scores, width, label='Tuned', alpha=0.7)

ax.set_xlabel('Metrics')
ax.set_ylabel('Scores')
ax.set_title('Logistic Regression: Baseline vs Tuned Performance')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
ax.grid(True, alpha=0.3)

# Add value labels on bars
for bar in bars1 + bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.3f}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),  # 3 points vertical offset
                textcoords="offset points",
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/logistic_regression/baseline_vs_tuned_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

#### 5.1.5 Save Results and Model

In [ ]:
# Enhanced Logistic Regression Results Saving with Comprehensive Error Handling
def save_logistic_regression_results():
    """Save all logistic regression results with comprehensive error handling"""
    
    output_dir = '../outputs/logistic_regression/'
    os.makedirs(output_dir, exist_ok=True)
    
    try:        
        # Save comprehensive performance metrics
        lr_performance_df = pd.DataFrame({
            'Model': ['Baseline_Logistic_Regression', 'Tuned_Logistic_Regression'],
            'Accuracy': [baseline_accuracy, tuned_accuracy],
            'Precision': [baseline_precision, tuned_precision],
            'Recall': [baseline_recall, tuned_recall],
            'F1_Score': [baseline_f1, tuned_f1],
            'AUC_ROC': [baseline_auc, tuned_auc]
        })
        lr_performance_df.to_csv(output_dir + 'performance_metrics.csv', index=False)
        
        # Save best parameters
        best_params_df = pd.DataFrame([grid_search.best_params_])
        best_params_df.to_csv(output_dir + 'best_parameters.csv', index=False)
        
        # Save tuned model feature importance (coefficients)
        feature_importance_tuned = pd.DataFrame({
            'Feature': X_train_final.columns,
            'Coefficient': best_lr.coef_[0],
            'Absolute_Value': np.abs(best_lr.coef_[0]),
            'Importance_Rank': np.argsort(np.abs(best_lr.coef_[0]))[::-1] + 1
        }).sort_values('Absolute_Value', ascending=False)
        
        feature_importance_tuned.to_csv(output_dir + 'feature_importance_tuned.csv', index=False)
        
        # Save predictions for detailed analysis
        lr_predictions_df = pd.DataFrame({
            'Actual': y_test_final,
            'Baseline_Predicted': y_pred_base,
            'Tuned_Predicted': y_pred_best,
            'Baseline_Probability': y_proba_base,
            'Tuned_Probability': y_proba_best,
            'Prediction_Correct': (y_test_final == y_pred_best).astype(int)
        })
        lr_predictions_df.to_csv(output_dir + 'predictions.csv', index=False)
        
        # Save the best model
        joblib.dump(best_lr, output_dir + 'best_logistic_regression_model.pkl')
        
        # Enhanced: Calculate additional metrics for model summary
        # Training accuracy for overfitting analysis
        y_train_pred_best = best_lr.predict(X_train_final)
        train_accuracy_best = accuracy_score(y_train_final, y_train_pred_best)
        overfitting_gap_lr = train_accuracy_best - tuned_accuracy
        
        # Cross-validation scores for robustness
        cv_scores_lr = cross_val_score(best_lr, X_train_final, y_train_final, cv=5, scoring='f1')
        
        # Feature importance insights
        top_3_features_lr = feature_importance_tuned.head(3)['Feature'].tolist()
        positive_coeff = (feature_importance_tuned['Coefficient'] > 0).sum()
        negative_coeff = (feature_importance_tuned['Coefficient'] < 0).sum()
        
        # Calculate odds ratios for interpretation
        odds_ratios = np.exp(feature_importance_tuned['Coefficient'])
        feature_importance_tuned['Odds_Ratio'] = odds_ratios
        
        # Save enhanced feature importance with odds ratios
        feature_importance_tuned.to_csv(output_dir + 'feature_importance_with_odds_ratios.csv', index=False)
        
        # Save comprehensive model summary
        model_summary_lr = {
            'best_accuracy': tuned_accuracy,
            'best_f1_score': tuned_f1,
            'best_auc_roc': tuned_auc,
            'best_precision': tuned_precision,
            'best_recall': tuned_recall,
            'training_accuracy': train_accuracy_best,
            'overfitting_gap': overfitting_gap_lr,
            'cv_f1_mean': cv_scores_lr.mean(),
            'cv_f1_std': cv_scores_lr.std(),
            'n_features': len(X_train_final.columns),
            'positive_coefficients': positive_coeff,
            'negative_coefficients': negative_coeff,
            'top_feature': top_3_features_lr[0],
            'top_feature_odds_ratio': odds_ratios.iloc[0],
            'intercept': best_lr.intercept_[0] if hasattr(best_lr, 'intercept_') else 'N/A'
        }
        
        model_summary_df = pd.DataFrame([model_summary_lr])
        model_summary_df.to_csv(output_dir + 'model_summary.csv', index=False)
        
        # Enhanced: Save detailed comparison between baseline and tuned
        comparison_df = pd.DataFrame({
            'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC', 'Training Accuracy', 'Overfitting Gap'],
            'Baseline_LR': [
                baseline_accuracy,
                baseline_precision, 
                baseline_recall,
                baseline_f1,
                baseline_auc,
                accuracy_score(y_train_final, baseline_lr.predict(X_train_final)) if 'baseline_lr' in locals() else 'N/A',
                (accuracy_score(y_train_final, baseline_lr.predict(X_train_final)) - baseline_accuracy) if 'baseline_lr' in locals() else 'N/A'
            ],
            'Tuned_LR': [
                tuned_accuracy,
                tuned_precision,
                tuned_recall,
                tuned_f1,
                tuned_auc,
                train_accuracy_best,
                overfitting_gap_lr
            ],
            'Improvement': [
                tuned_accuracy - baseline_accuracy,
                tuned_precision - baseline_precision,
                tuned_recall - baseline_recall,
                tuned_f1 - baseline_f1,
                tuned_auc - baseline_auc,
                train_accuracy_best - (accuracy_score(y_train_final, baseline_lr.predict(X_train_final)) if 'baseline_lr' in locals() else 0),
                overfitting_gap_lr - ((accuracy_score(y_train_final, baseline_lr.predict(X_train_final)) - baseline_accuracy) if 'baseline_lr' in locals() else 0)
            ]
        })
        comparison_df.to_csv(output_dir + 'detailed_comparison.csv', index=False)
        
        # Enhanced: Save coefficient analysis for model interpretation
        coefficient_analysis = feature_importance_tuned.copy()
        coefficient_analysis['Effect_Direction'] = coefficient_analysis['Coefficient'].apply(
            lambda x: 'Positive (Increases Risk)' if x > 0 else 'Negative (Decreases Risk)'
        )
        coefficient_analysis['Effect_Strength'] = coefficient_analysis['Absolute_Value'].apply(
            lambda x: 'Strong' if x > 0.5 else 'Medium' if x > 0.1 else 'Weak'
        )
        coefficient_analysis.to_csv(output_dir + 'coefficient_analysis.csv', index=False)
        
        print(f"\nALL LOGISTIC REGRESSION RESULTS SAVED SUCCESSFULLY IN '{output_dir}'")
        
        # Print summary statistics
        print(f"\nLOGISTIC REGRESSION MODEL SUMMARY:")
        print(f"   Best Accuracy: {tuned_accuracy:.4f} ({tuned_accuracy*100:.2f}%)")
        print(f"   Best F1-Score: {tuned_f1:.4f}")
        print(f"   Best AUC-ROC: {tuned_auc:.4f}")
        print(f"   Overfitting Gap: {overfitting_gap_lr:.4f}")
        print(f"   Top Feature: {top_3_features_lr[0]} (Odds Ratio: {odds_ratios.iloc[0]:.4f})")
        print(f"   Positive Coefficients: {positive_coeff}")
        print(f"   Negative Coefficients: {negative_coeff}")
        
        # Interpretation of top feature
        top_feature_effect = "increases" if feature_importance_tuned.iloc[0]['Coefficient'] > 0 else "decreases"
        print(f"   Interpretation: {top_3_features_lr[0]} {top_feature_effect} disease risk")
        
    except Exception as e:
        print(f"ERROR saving Logistic Regression results: {e}")
        import traceback
        print("Detailed error traceback:")
        traceback.print_exc()
        
# Execute the enhanced saving function
save_logistic_regression_results()

#### 5.1.6 Visualization and Comprehensive Analysis

In [ ]:
# Confusion Matrix for Tuned Logistic Regression
cm_tuned = confusion_matrix(y_test_final, y_pred_best)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_tuned, annot=True, fmt='d', cmap='Greens', 
            xticklabels=['No Disease', 'Disease'],
            yticklabels=['No Disease', 'Disease'])
plt.title('Confusion Matrix - Tuned Logistic Regression')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('../outputs/logistic_regression/confusion_matrix_tuned.png', dpi=300, bbox_inches='tight')
plt.show()

# ROC Curve Comparison (Baseline vs Tuned)
fpr_tuned, tpr_tuned, _ = roc_curve(y_test_final, y_proba_best)

plt.figure(figsize=(8, 6))
plt.plot(fpr_base, tpr_base, label=f'Baseline LR (AUC = {baseline_auc:.3f})', linewidth=2)
plt.plot(fpr_tuned, tpr_tuned, label=f'Tuned LR (AUC = {tuned_auc:.3f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Logistic Regression Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/logistic_regression/roc_curve_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nLogistic Regression analysis completed and all results saved!")

### 5.2 Decision Trees implementation & tuning

#### 5.2.1 Baseline Decision Tree with Cross-Validation

In [ ]:
# Define baseline model
baseline_dt = DecisionTreeClassifier(random_state=42)

print("Performing 5-fold cross-validation for baseline Decision Tree...")
cv_scores_dt = cross_val_score(baseline_dt, X_train_final, y_train_final, cv=5, scoring='f1')
print(f"Baseline DT Cross-Validation F1 Scores: {cv_scores_dt}")
print(f"Baseline DT CV F1 Mean: {cv_scores_dt.mean():.4f} (+/- {cv_scores_dt.std() * 2:.4f})")

# Additional cross-validation with different metrics
print("\nAdditional cross-validation metrics:")
cv_accuracy_dt = cross_val_score(baseline_dt, X_train_final, y_train_final, cv=5, scoring='accuracy')
cv_precision_dt = cross_val_score(baseline_dt, X_train_final, y_train_final, cv=5, scoring='precision')
cv_recall_dt = cross_val_score(baseline_dt, X_train_final, y_train_final, cv=5, scoring='recall')
cv_auc_dt = cross_val_score(baseline_dt, X_train_final, y_train_final, cv=5, scoring='roc_auc')

print(f"CV Accuracy:  {cv_accuracy_dt.mean():.4f} (+/- {cv_accuracy_dt.std() * 2:.4f})")
print(f"CV Precision: {cv_precision_dt.mean():.4f} (+/- {cv_precision_dt.std() * 2:.4f})")
print(f"CV Recall:    {cv_recall_dt.mean():.4f} (+/- {cv_recall_dt.std() * 2:.4f})")
print(f"CV AUC-ROC:   {cv_auc_dt.mean():.4f} (+/- {cv_auc_dt.std() * 2:.4f})")

# Now fit the baseline model on full training data
print("\nTraining baseline Decision Tree on full training set...")
baseline_dt.fit(X_train_final, y_train_final)

# Evaluate baseline Decision Tree on test set
print("Evaluating baseline Decision Tree on test set...")
y_pred_dt_base = baseline_dt.predict(X_test_final)
y_proba_dt_base = baseline_dt.predict_proba(X_test_final)[:, 1]

# Calculate baseline Decision Tree performance metrics
baseline_dt_accuracy = accuracy_score(y_test_final, y_pred_dt_base)
baseline_dt_precision = precision_score(y_test_final, y_pred_dt_base)
baseline_dt_recall = recall_score(y_test_final, y_pred_dt_base)
baseline_dt_f1 = f1_score(y_test_final, y_pred_dt_base)
baseline_dt_auc = roc_auc_score(y_test_final, y_proba_dt_base)

print("\nBaseline Decision Tree Performance (Test Set):")
print(f"Accuracy:  {baseline_dt_accuracy:.4f}")
print(f"Precision: {baseline_dt_precision:.4f}")
print(f"Recall:    {baseline_dt_recall:.4f}")
print(f"F1-Score:  {baseline_dt_f1:.4f}")
print(f"AUC-ROC:   {baseline_dt_auc:.4f}")

# Confusion Matrix for Baseline
cm_dt_base = confusion_matrix(y_test_final, y_pred_dt_base)
print(f"\nConfusion Matrix (Baseline):")
print(f"True Negatives:  {cm_dt_base[0,0]}")
print(f"False Positives: {cm_dt_base[0,1]}")
print(f"False Negatives: {cm_dt_base[1,0]}")
print(f"True Positives:  {cm_dt_base[1,1]}")

# Plot baseline confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm_dt_base, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No Disease', 'Disease'],
            yticklabels=['No Disease', 'Disease'])
plt.title('Confusion Matrix - Baseline Decision Tree')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('../outputs/decision_tree/confusion_matrix_baseline.png', dpi=300, bbox_inches='tight')
plt.show()

print("BASELINE TREE ANALYSIS")

print(f"Tree Depth: {baseline_dt.get_depth()}")
print(f"Number of Leaves: {baseline_dt.get_n_leaves()}")
print(f"Number of Features Used: {baseline_dt.n_features_in_}")

# Check for overfitting in baseline
y_train_pred_base = baseline_dt.predict(X_train_final)
train_accuracy_base = accuracy_score(y_train_final, y_train_pred_base)
overfitting_gap_base = train_accuracy_base - baseline_dt_accuracy

print(f"Training Accuracy: {train_accuracy_base:.4f}")
print(f"Testing Accuracy: {baseline_dt_accuracy:.4f}")
print(f"Overfitting Gap: {overfitting_gap_base:.4f}")

if overfitting_gap_base > 0.1:
    print("WARNING: Significant overfitting in baseline model!")
else:
    print("Baseline model shows acceptable overfitting")

 #### 5.2.2 Hyperparameter Tuning with GridSearchCV

In [ ]:
print("HYPERPARAMETER TUNING FOR DECISION TREE")

param_grid_dt = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [3, 5, 7, 10, 15, None],
    'min_samples_split': [2, 5, 10, 15],
    'min_samples_leaf': [1, 2, 4, 6],
    'max_features': ['sqrt', 'log2', None]
}

print("Starting GridSearchCV for Decision Tree...")
grid_search_dt = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid_dt,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)
grid_search_dt.fit(X_train_final, y_train_final)

print(f"\nBest parameters found: {grid_search_dt.best_params_}")
print(f"Best cross-validation score: {grid_search_dt.best_score_:.4f}")

#### 5.2.3 Train Best Model and Evaluate

In [ ]:
# Train Best Model and Evaluate
best_dt = grid_search_dt.best_estimator_

# Evaluate tuned Decision Tree
y_pred_dt_best = best_dt.predict(X_test_final)
y_proba_dt_best = best_dt.predict_proba(X_test_final)[:, 1]

# Tuned Decision Tree performance
tuned_dt_accuracy = accuracy_score(y_test_final, y_pred_dt_best)
tuned_dt_precision = precision_score(y_test_final, y_pred_dt_best)
tuned_dt_recall = recall_score(y_test_final, y_pred_dt_best)
tuned_dt_f1 = f1_score(y_test_final, y_pred_dt_best)
tuned_dt_auc = roc_auc_score(y_test_final, y_proba_dt_best)

print("\nTuned Decision Tree Performance:")
print(f"Accuracy:  {tuned_dt_accuracy:.4f}")
print(f"Precision: {tuned_dt_precision:.4f}")
print(f"Recall:    {tuned_dt_recall:.4f}")
print(f"F1-Score:  {tuned_dt_f1:.4f}")
print(f"AUC-ROC:   {tuned_dt_auc:.4f}")

print(f"\nImprovement over baseline:")
print(f"Accuracy:  {tuned_dt_accuracy - baseline_dt_accuracy:+.4f}")
print(f"F1-Score:  {tuned_dt_f1 - baseline_dt_f1:+.4f}")

#### 5.2.4 Feature Importance Analysis

In [ ]:
feature_importance_dt = pd.DataFrame({
    'Feature': X_train_final.columns,
    'Importance': best_dt.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nTop 10 Most Important Features (Decision Tree):")
print(feature_importance_dt.head(10))

# Plot feature importance
plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance_dt.head(10), x='Importance', y='Feature')
plt.title('Top 10 Feature Importances - Decision Tree')
plt.tight_layout()
plt.savefig('../outputs/decision_tree/feature_importance_dt.png', dpi=300, bbox_inches='tight')
plt.show()

#### 5.2.5 Tree Analysis

In [ ]:
print("OPTIMIZED TREE ANALYSIS")

print(f"Optimal Tree Depth: {best_dt.get_depth()}")
print(f"Number of Leaves: {best_dt.get_n_leaves()}")
print(f"Number of Features Used: {best_dt.n_features_in_}")

# Check overfitting after tuning
y_train_pred_tuned = best_dt.predict(X_train_final)
train_accuracy_tuned = accuracy_score(y_train_final, y_train_pred_tuned)
overfitting_gap_tuned = train_accuracy_tuned - tuned_dt_accuracy

print(f"Training Accuracy (Tuned): {train_accuracy_tuned:.4f}")
print(f"Testing Accuracy (Tuned): {tuned_dt_accuracy:.4f}")
print(f"Overfitting Gap (Tuned): {overfitting_gap_tuned:.4f}")

# Cross-validation score for tuned model robustness
cv_scores_tuned = cross_val_score(best_dt, X_train_final, y_train_final, cv=5, scoring='f1')
print(f"\nTuned Model Cross-Validation:")
print(f"CV F1 Scores: {[f'{score:.4f}' for score in cv_scores_tuned]}")
print(f"Mean CV F1 Score: {cv_scores_tuned.mean():.4f} (+/- {cv_scores_tuned.std() * 2:.4f})")

if overfitting_gap_tuned < 0.05:
    print("EXCELLENT: Minimal overfitting after tuning!")
elif overfitting_gap_tuned < 0.1:
    print("MODERATE: Some overfitting still present")
else:
    print("SIGNIFICANT: Considerable overfitting detected")


#### 5.2.6 Final Comprehensive Comparison

In [ ]:
print("FINAL COMPREHENSIVE PERFORMANCE COMPARISON")

print(f"\n{'Metric':<20} {'Baseline':<12} {'Tuned':<12} {'Improvement':<15}")
print("-" * 65)

print(f"{'Accuracy':<20} {baseline_dt_accuracy:<12.4f} {tuned_dt_accuracy:<12.4f} {tuned_dt_accuracy - baseline_dt_accuracy:+.4f}")
print(f"{'F1-Score':<20} {baseline_dt_f1:<12.4f} {tuned_dt_f1:<12.4f} {tuned_dt_f1 - baseline_dt_f1:+.4f}")
print(f"{'Precision':<20} {baseline_dt_precision:<12.4f} {tuned_dt_precision:<12.4f} {tuned_dt_precision - baseline_dt_precision:+.4f}")
print(f"{'Recall':<20} {baseline_dt_recall:<12.4f} {tuned_dt_recall:<12.4f} {tuned_dt_recall - baseline_dt_recall:+.4f}")
print(f"{'AUC-ROC':<20} {baseline_dt_auc:<12.4f} {tuned_dt_auc:<12.4f} {tuned_dt_auc - baseline_dt_auc:+.4f}")
print(f"{'Tree Depth':<20} {baseline_dt.get_depth():<12} {best_dt.get_depth():<12} {best_dt.get_depth() - baseline_dt.get_depth():+d}")
print(f"{'Overfitting Gap':<20} {overfitting_gap_base:<12.4f} {overfitting_gap_tuned:<12.4f} {overfitting_gap_tuned - overfitting_gap_base:+.4f}")

# Overall improvement percentages
accuracy_improvement_pct = ((tuned_dt_accuracy - baseline_dt_accuracy) / baseline_dt_accuracy) * 100
overfitting_reduction_pct = ((overfitting_gap_base - overfitting_gap_tuned) / overfitting_gap_base) * 100

print(f"\nOVERALL IMPROVEMENT SUMMARY:")
print(f"    Accuracy Improvement: {accuracy_improvement_pct:+.1f}%")
print(f"    Overfitting Reduction: {overfitting_reduction_pct:+.1f}%")
print(f"    Tree Complexity: Reduced from {baseline_dt.get_depth()} to {best_dt.get_depth()} levels")

# Top features analysis
top_3_features = feature_importance_dt.head(3)['Feature'].tolist()
print(f"    Top 3 Features: {top_3_features}")
print(f"    Features Used: {(feature_importance_dt['Importance'] > 0).sum()} out of {len(feature_importance_dt)}")

#### 5.2.7 Save Results and Model

In [ ]:
def save_decision_tree_results():
    """Save all decision tree results with comprehensive error handling"""
    
    output_dir = '../outputs/decision_tree/'
    os.makedirs(output_dir, exist_ok=True)
    
    try:
        # Save performance metrics comparison
        performance_df = pd.DataFrame({
            'Model': ['Baseline_DT', 'Tuned_DT'],
            'Accuracy': [baseline_dt_accuracy, tuned_dt_accuracy],
            'F1_Score': [baseline_dt_f1, tuned_dt_f1],
            'Precision': [baseline_dt_precision, tuned_dt_precision],
            'Recall': [baseline_dt_recall, tuned_dt_recall],
            'AUC_ROC': [baseline_dt_auc, tuned_dt_auc],
            'Tree_Depth': [baseline_dt.get_depth(), best_dt.get_depth()],
            'Overfitting_Gap': [overfitting_gap_base, overfitting_gap_tuned]
        })
        performance_df.to_csv(output_dir + 'performance_comparison.csv', index=False)
        
        # Save best parameters
        best_params_df = pd.DataFrame([grid_search_dt.best_params_])
        best_params_df.to_csv(output_dir + 'best_parameters.csv', index=False)
        
        # Save feature importance
        feature_importance_dt.to_csv(output_dir + 'feature_importance.csv', index=False)
        
        # Save predictions
        predictions_df = pd.DataFrame({
            'Actual': y_test_final.values,
            'Baseline_Predicted': y_pred_dt_base,
            'Tuned_Predicted': y_pred_dt_best,
            'Baseline_Probability': y_proba_dt_base,
            'Tuned_Probability': y_proba_dt_best
        })
        predictions_df.to_csv(output_dir + 'predictions_comparison.csv', index=False)
        
        # Save the trained model
        joblib.dump(best_dt, output_dir + 'best_decision_tree_model.pkl')
        
        # Save model summary
        summary = {
            'best_accuracy': tuned_dt_accuracy,
            'best_f1_score': tuned_dt_f1,
            'best_auc_roc': tuned_dt_auc,
            'tree_depth': best_dt.get_depth(),
            'n_leaves': best_dt.get_n_leaves(),
            'top_features': feature_importance_dt.head(3)['Feature'].tolist(),
            'overfitting_gap': overfitting_gap_tuned,
            'cv_f1_mean': cv_scores_tuned.mean(),
            'cv_f1_std': cv_scores_tuned.std()
        }
        pd.DataFrame([summary]).to_csv(output_dir + 'model_summary.csv', index=False)
        
        print(f"\nALL RESULTS SAVED SUCCESSFULLY in: {output_dir}")
        
    except Exception as e:
        print(f"Error saving results: {e}")
        import traceback
        traceback.print_exc()

# Execute saving function
save_decision_tree_results()

#### 5.2.8 Visualization and Model Analysis

In [ ]:
# Confusion Matrix for Tuned Decision Tree
cm_dt_tuned = confusion_matrix(y_test_final, y_pred_dt_best)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_dt_tuned, annot=True, fmt='d', cmap='Greens', 
            xticklabels=['No Disease', 'Disease'],
            yticklabels=['No Disease', 'Disease'])
plt.title('Confusion Matrix - Tuned Decision Tree')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('../outputs/decision_tree/confusion_matrix_tuned.png', dpi=300, bbox_inches='tight')
plt.show()

# ROC Curve Comparison
fpr_dt_base, tpr_dt_base, _ = roc_curve(y_test_final, y_proba_dt_base)
fpr_dt_tuned, tpr_dt_tuned, _ = roc_curve(y_test_final, y_proba_dt_best)

plt.figure(figsize=(8, 6))
plt.plot(fpr_dt_base, tpr_dt_base, label=f'Baseline DT (AUC = {baseline_dt_auc:.3f})', linewidth=2)
plt.plot(fpr_dt_tuned, tpr_dt_tuned, label=f'Tuned DT (AUC = {tuned_dt_auc:.3f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Decision Tree')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/decision_tree/roc_curve_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

# Performance Comparison Plot
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
baseline_dt_scores = [baseline_dt_accuracy, baseline_dt_precision, baseline_dt_recall, baseline_dt_f1, baseline_dt_auc]
tuned_dt_scores = [tuned_dt_accuracy, tuned_dt_precision, tuned_dt_recall, tuned_dt_f1, tuned_dt_auc]

plt.figure(figsize=(12, 6))
x = np.arange(len(metrics))
width = 0.35

bars1 = plt.bar(x - width/2, baseline_dt_scores, width, label='Baseline DT', alpha=0.7, color='skyblue')
bars2 = plt.bar(x + width/2, tuned_dt_scores, width, label='Tuned DT', alpha=0.7, color='lightcoral')

plt.xlabel('Metrics')
plt.ylabel('Scores')
plt.title('Decision Tree: Baseline vs Tuned Performance')
plt.xticks(x, metrics)
plt.legend()
plt.grid(True, alpha=0.3)

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.savefig('../outputs/decision_tree/baseline_vs_tuned_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

#### 5.2.9 Tree Visualization and Complexity Analysis

In [ ]:
# Tree Visualization (simplified for readability)
plt.figure(figsize=(16, 10))
plot_tree(best_dt,
          feature_names=X_train_final.columns,
          class_names=['No Disease', 'Disease'],
          filled=True,
          rounded=True,
          fontsize=10,
          max_depth=3,  # Limit depth for readability
          proportion=True,
          impurity=False)
plt.title('Optimized Decision Tree - Liver Disease Classification', fontsize=14, pad=20)
plt.tight_layout()
plt.savefig('../outputs/decision_tree/tree_visualization.png', dpi=300, bbox_inches='tight')
plt.show()

print("Decision Tree Visualization Completed!")
print(f"   - Tree Depth: {best_dt.get_depth()}")
print(f"   - Number of Leaves: {best_dt.get_n_leaves()}")
print(f"   - Most important feature: {feature_importance_dt.iloc[0]['Feature']}")

#### 5.2.10 Additional Model Insights

In [ ]:
# Classification report for tuned model
print("\nDetailed Classification Report (Tuned Decision Tree):")
print(classification_report(y_test_final, y_pred_dt_best, target_names=['No Disease', 'Disease']))

# Feature importance insights
print("\nFeature Importance Insights:")
for i, row in feature_importance_dt.head(5).iterrows():
    print(f"  {row['Feature']}: {row['Importance']:.4f}")

#### 5.2.11 Final Summary

In [ ]:
print("DECISION TREE IMPLEMENTATION COMPLETED SUCCESSFULLY!")

print(f"Final Tuned Model Performance:")
print(f"  Accuracy: {tuned_dt_accuracy:.4f} ({tuned_dt_accuracy*100:.2f}%)")
print(f"  F1-Score: {tuned_dt_f1:.4f}")
print(f"  AUC-ROC:  {tuned_dt_auc:.4f}")
print(f"  Tree Complexity: {best_dt.get_depth()} levels, {best_dt.get_n_leaves()} leaves")
print(f"  Overfitting Gap: {overfitting_gap_tuned:.4f}")

print(f"\nAll results saved to: ../outputs/decision_tree/")
print(f"Best model saved as: best_decision_tree_model.pkl")

### 5.3 Random Forests implementation & tuning

#### 5.3.1 Baseline Random Forest

In [ ]:
# Define baseline model
baseline_rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

print("Performing 5-fold cross-validation for baseline Random Forest...")
cv_scores_rf = cross_val_score(baseline_rf, X_train_final, y_train_final, cv=5, scoring='f1')
print(f"Baseline RF Cross-Validation F1 Scores: {cv_scores_rf}")
print(f"Baseline RF CV F1 Mean: {cv_scores_rf.mean():.4f} (+/- {cv_scores_rf.std() * 2:.4f})")

# Additional cross-validation with different metrics for comprehensive analysis
print("\nAdditional cross-validation metrics:")
cv_accuracy = cross_val_score(baseline_rf, X_train_final, y_train_final, cv=5, scoring='accuracy')
cv_precision = cross_val_score(baseline_rf, X_train_final, y_train_final, cv=5, scoring='precision')
cv_recall = cross_val_score(baseline_rf, X_train_final, y_train_final, cv=5, scoring='recall')
cv_auc = cross_val_score(baseline_rf, X_train_final, y_train_final, cv=5, scoring='roc_auc')

print(f"CV Accuracy:  {cv_accuracy.mean():.4f} (+/- {cv_accuracy.std() * 2:.4f})")
print(f"CV Precision: {cv_precision.mean():.4f} (+/- {cv_precision.std() * 2:.4f})")
print(f"CV Recall:    {cv_recall.mean():.4f} (+/- {cv_recall.std() * 2:.4f})")
print(f"CV AUC-ROC:   {cv_auc.mean():.4f} (+/- {cv_auc.std() * 2:.4f})")

# Now fit the baseline model on full training data for final evaluation
print("\nTraining baseline Random Forest on full training set...")
baseline_rf.fit(X_train_final, y_train_final)

# Evaluate baseline Random Forest on test set
print("Evaluating baseline Random Forest on test set...")
y_pred_rf_base = baseline_rf.predict(X_test_final)
y_proba_rf_base = baseline_rf.predict_proba(X_test_final)[:, 1]

# Calculate baseline Random Forest performance metrics
baseline_rf_accuracy = accuracy_score(y_test_final, y_pred_rf_base)
baseline_rf_precision = precision_score(y_test_final, y_pred_rf_base)
baseline_rf_recall = recall_score(y_test_final, y_pred_rf_base)
baseline_rf_f1 = f1_score(y_test_final, y_pred_rf_base)
baseline_rf_auc = roc_auc_score(y_test_final, y_proba_rf_base)

print("\nBaseline Random Forest Performance (Test Set):")
print("-" * 50)
print(f"Accuracy:  {baseline_rf_accuracy:.4f}")
print(f"Precision: {baseline_rf_precision:.4f}")
print(f"Recall:    {baseline_rf_recall:.4f}")
print(f"F1-Score:  {baseline_rf_f1:.4f}")
print(f"AUC-ROC:   {baseline_rf_auc:.4f}")

# Confusion Matrix for Baseline
cm_rf_base = confusion_matrix(y_test_final, y_pred_rf_base)
print(f"\nConfusion Matrix (Baseline):")
print(f"True Negatives:  {cm_rf_base[0,0]}")
print(f"False Positives: {cm_rf_base[0,1]}")
print(f"False Negatives: {cm_rf_base[1,0]}")
print(f"True Positives:  {cm_rf_base[1,1]}")

# Plot baseline confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm_rf_base, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No Disease', 'Disease'],
            yticklabels=['No Disease', 'Disease'])
plt.title('Confusion Matrix - Baseline Random Forest')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('../outputs/random_forest/confusion_matrix_baseline.png', dpi=300, bbox_inches='tight')
plt.show()

# Feature Importance for Baseline
feature_importance_base_rf = pd.DataFrame({
    'Feature': X_train_final.columns,
    'Importance': baseline_rf.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nTop 10 Most Important Features (Baseline Random Forest):")
print(feature_importance_base_rf.head(10).round(4))

# Plot baseline feature importance
plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance_base_rf.head(10), 
            x='Importance', 
            y='Feature', 
            hue='Feature',  # Added hue parameter
            legend=False,   # Disable legend
            palette='viridis')
plt.title('Top 10 Feature Importances - Baseline Random Forest')
plt.tight_layout()
plt.savefig('../outputs/random_forest/feature_importance_baseline.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nBaseline Random Forest analysis completed!")

#### 5.3.2 Hyperparameter Tuning with RandomizedSearchCV

In [ ]:
param_dist_rf = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [None, 10, 20, 30, 50],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None],
    'bootstrap': [True, False],
    'class_weight': [None, 'balanced', 'balanced_subsample']
}

random_search_rf = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_dist_rf,
    n_iter=50,
    cv=5,
    scoring='f1',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("Starting RandomizedSearchCV for Random Forest...")
random_search_rf.fit(X_train_final, y_train_final)

print("\nBest parameters found:")
print(random_search_rf.best_params_)
print(f"Best cross-validation score: {random_search_rf.best_score_:.4f}")

#### 5.3.3 Train Best Model and Evaluate

In [ ]:
best_rf = random_search_rf.best_estimator_

# Evaluate tuned Random Forest
y_pred_rf_best = best_rf.predict(X_test_final)
y_proba_rf_best = best_rf.predict_proba(X_test_final)[:, 1]

# Tuned Random Forest performance
tuned_rf_accuracy = accuracy_score(y_test_final, y_pred_rf_best)
tuned_rf_precision = precision_score(y_test_final, y_pred_rf_best)
tuned_rf_recall = recall_score(y_test_final, y_pred_rf_best)
tuned_rf_f1 = f1_score(y_test_final, y_pred_rf_best)
tuned_rf_auc = roc_auc_score(y_test_final, y_proba_rf_best)

print("\nTuned Random Forest Performance:")
print(f"Accuracy:  {tuned_rf_accuracy:.4f}")
print(f"Precision: {tuned_rf_precision:.4f}")
print(f"Recall:    {tuned_rf_recall:.4f}")
print(f"F1-Score:  {tuned_rf_f1:.4f}")
print(f"AUC-ROC:   {tuned_rf_auc:.4f}")

print(f"\nImprovement over baseline:")
print(f"Accuracy:  {tuned_rf_accuracy - baseline_rf_accuracy:+.4f}")
print(f"F1-Score:  {tuned_rf_f1 - baseline_rf_f1:+.4f}")

#### 5.3.4 Feature Importance Analysis

In [ ]:
# Get feature importance
feature_importance_rf = pd.DataFrame({
    'Feature': X_train_final.columns,
    'Importance': best_rf.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nTop 10 Most Important Features (Random Forest):")
print(feature_importance_rf.head(10))

# Plot feature importance
plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance_rf.head(10), x='Importance', y='Feature')
plt.title('Top 10 Feature Importances - Random Forest')
plt.tight_layout()
plt.savefig('../outputs/random_forest/feature_importance_rf.png', dpi=300, bbox_inches='tight')
plt.show()

#### 5.3.5 Save Results and Model

In [ ]:
def save_random_forest_results():
    """Save all random forest results with comprehensive error handling"""
    
    output_dir = '../outputs/random_forest/'
    os.makedirs(output_dir, exist_ok=True)
    
    try:        
        # Save performance metrics
        rf_performance_df = pd.DataFrame({
            'Model': ['Baseline_RF', 'Tuned_RF'],
            'Accuracy': [baseline_rf_accuracy, tuned_rf_accuracy],
            'Precision': [baseline_rf_precision, tuned_rf_precision],
            'Recall': [baseline_rf_recall, tuned_rf_recall],
            'F1_Score': [baseline_rf_f1, tuned_rf_f1],
            'AUC_ROC': [baseline_rf_auc, tuned_rf_auc]
        })
        rf_performance_df.to_csv(output_dir + 'performance_metrics.csv', index=False)
        
        # Save best parameters
        best_params_rf_df = pd.DataFrame([random_search_rf.best_params_])
        best_params_rf_df.to_csv(output_dir + 'best_parameters.csv', index=False)
        
        # Save feature importance
        feature_importance_rf.to_csv(output_dir + 'feature_importance.csv', index=False)
        
        # Save predictions
        rf_predictions_df = pd.DataFrame({
            'Actual': y_test_final,
            'Baseline_Predicted': y_pred_rf_base,
            'Tuned_Predicted': y_pred_rf_best,
            'Baseline_Probability': y_proba_rf_base,
            'Tuned_Probability': y_proba_rf_best
        })
        rf_predictions_df.to_csv(output_dir + 'predictions.csv', index=False)
        
        # Save the trained model
        joblib.dump(best_rf, output_dir + 'best_random_forest_model.pkl')
        
        # Save model summary with additional metrics
        # Calculate training accuracy for overfitting analysis
        y_train_pred_rf_best = best_rf.predict(X_train_final)
        train_accuracy_rf_best = accuracy_score(y_train_final, y_train_pred_rf_best)
        overfitting_gap_rf = train_accuracy_rf_best - tuned_rf_accuracy
        
        # Calculate feature importance insights
        top_3_features_rf = feature_importance_rf.head(3)['Feature'].tolist()
        features_used_rf = (feature_importance_rf['Importance'] > 0).sum()
        
        # Cross-validation scores for robustness
        cv_scores_rf = cross_val_score(best_rf, X_train_final, y_train_final, cv=5, scoring='f1')
        
        # Save comprehensive model summary
        model_summary_rf = {
            'best_accuracy': tuned_rf_accuracy,
            'best_f1_score': tuned_rf_f1,
            'best_auc_roc': tuned_rf_auc,
            'n_estimators': best_rf.n_estimators,
            'max_depth': best_rf.max_depth if best_rf.max_depth is not None else 'None',
            'top_features': top_3_features_rf,
            'features_used': features_used_rf,
            'overfitting_gap': overfitting_gap_rf,
            'cv_f1_mean': cv_scores_rf.mean(),
            'cv_f1_std': cv_scores_rf.std(),
            'training_accuracy': train_accuracy_rf_best,
            'test_accuracy': tuned_rf_accuracy
        }
        
        model_summary_df = pd.DataFrame([model_summary_rf])
        model_summary_df.to_csv(output_dir + 'model_summary.csv', index=False)
        
        # Save comprehensive comparison between baseline and tuned
        comparison_df = pd.DataFrame({
            'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC', 'Training Accuracy', 'Overfitting Gap'],
            'Baseline_RF': [
                baseline_rf_accuracy,
                baseline_rf_precision, 
                baseline_rf_recall,
                baseline_rf_f1,
                baseline_rf_auc,
                accuracy_score(y_train_final, baseline_rf.predict(X_train_final)),
                accuracy_score(y_train_final, baseline_rf.predict(X_train_final)) - baseline_rf_accuracy
            ],
            'Tuned_RF': [
                tuned_rf_accuracy,
                tuned_rf_precision,
                tuned_rf_recall,
                tuned_rf_f1,
                tuned_rf_auc,
                train_accuracy_rf_best,
                overfitting_gap_rf
            ],
            'Improvement': [
                tuned_rf_accuracy - baseline_rf_accuracy,
                tuned_rf_precision - baseline_rf_precision,
                tuned_rf_recall - baseline_rf_recall,
                tuned_rf_f1 - baseline_rf_f1,
                tuned_rf_auc - baseline_rf_auc,
                train_accuracy_rf_best - accuracy_score(y_train_final, baseline_rf.predict(X_train_final)),
                overfitting_gap_rf - (accuracy_score(y_train_final, baseline_rf.predict(X_train_final)) - baseline_rf_accuracy)
            ]
        })
        comparison_df.to_csv(output_dir + 'detailed_comparison.csv', index=False)
        
        print(f"\nALL RANDOM FOREST RESULTS SAVED SUCCESSFULLY in: {output_dir}")

        # Print summary statistics
        print(f"\nMODEL SUMMARY:")
        print(f"   Best Accuracy: {tuned_rf_accuracy:.4f} ({tuned_rf_accuracy*100:.2f}%)")
        print(f"   Best F1-Score: {tuned_rf_f1:.4f}")
        print(f"   Best AUC-ROC: {tuned_rf_auc:.4f}")
        print(f"   Overfitting Gap: {overfitting_gap_rf:.4f}")
        print(f"   Top Feature: {top_3_features_rf[0]}")
        
    except Exception as e:
        print(f"ERROR saving Random Forest results: {e}")
        import traceback
        print("Detailed error traceback:")
        traceback.print_exc()
        
        # Try to save at least the critical components even if others fail
        try:
            print("\nAttempting to save critical components...")
            # Save model as it's most important
            joblib.dump(best_rf, output_dir + 'best_random_forest_model.pkl')
            print("Critical: Model saved despite other errors")
        except Exception as critical_error:
            print(f"CRITICAL: Could not save model: {critical_error}")

# Execute the enhanced saving function
save_random_forest_results()

#### 5.3.6 Visualization and Model Analysis

In [ ]:
# Confusion Matrix for Tuned Random Forest
cm_rf = confusion_matrix(y_test_final, y_pred_rf_best)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens', 
            xticklabels=['No Disease', 'Disease'],
            yticklabels=['No Disease', 'Disease'])
plt.title('Confusion Matrix - Tuned Random Forest')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('../outputs/random_forest/confusion_matrix_tuned.png', dpi=300, bbox_inches='tight')
plt.show()

# ROC Curve Comparison
fpr_rf_base, tpr_rf_base, _ = roc_curve(y_test_final, y_proba_rf_base)
fpr_rf_tuned, tpr_rf_tuned, _ = roc_curve(y_test_final, y_proba_rf_best)

plt.figure(figsize=(8, 6))
plt.plot(fpr_rf_base, tpr_rf_base, label=f'Baseline RF (AUC = {baseline_rf_auc:.3f})', linewidth=2)
plt.plot(fpr_rf_tuned, tpr_rf_tuned, label=f'Tuned RF (AUC = {tuned_rf_auc:.3f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Random Forest')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/random_forest/roc_curve_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

# Performance Comparison Plot
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
baseline_rf_scores = [baseline_rf_accuracy, baseline_rf_precision, baseline_rf_recall, baseline_rf_f1, baseline_rf_auc]
tuned_rf_scores = [tuned_rf_accuracy, tuned_rf_precision, tuned_rf_recall, tuned_rf_f1, tuned_rf_auc]

plt.figure(figsize=(12, 6))
x = np.arange(len(metrics))
width = 0.35

bars1 = plt.bar(x - width/2, baseline_rf_scores, width, label='Baseline RF', alpha=0.7, color='skyblue')
bars2 = plt.bar(x + width/2, tuned_rf_scores, width, label='Tuned RF', alpha=0.7, color='lightcoral')

plt.xlabel('Metrics')
plt.ylabel('Scores')
plt.title('Random Forest: Baseline vs Tuned Performance')
plt.xticks(x, metrics)
plt.legend()
plt.grid(True, alpha=0.3)

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.savefig('../outputs/random_forest/baseline_vs_tuned_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## Task 6: Model comparison

### 6.1 Load performance result files for all models

In [ ]:
# Paths to model results saved in the project structure
lr_perf_path = "../outputs/logistic_regression/performance_metrics.csv"
dt_perf_path = "../outputs/decision_tree/performance_comparison.csv"
rf_perf_path = "../outputs/random_forest/performance_metrics.csv"

# Read CSV files into DataFrames
lr_perf = pd.read_csv(lr_perf_path)
dt_perf = pd.read_csv(dt_perf_path)
rf_perf = pd.read_csv(rf_perf_path)

print("Files loaded successfully.")
print("Logistic Regression rows:", lr_perf.shape[0])
print("Decision Tree rows:", dt_perf.shape[0])
print("Random Forest rows:", rf_perf.shape[0])


### 6.2 Select tuned model rows

In [ ]:
lr_tuned = lr_perf[lr_perf["Model"] == "Tuned_Logistic_Regression"].copy()
dt_tuned = dt_perf[dt_perf["Model"] == "Tuned_DT"].copy()
rf_tuned = rf_perf[rf_perf["Model"] == "Tuned_RF"].copy()

print("Tuned Logistic Regression row:")
display(lr_tuned)

print("Tuned Decision Tree row:")
display(dt_tuned)

print("Tuned Random Forest row:")
display(rf_tuned)

### 6.3 Build unified comparison table

In [ ]:
comparison_df = pd.DataFrame({
"Model": [
"Logistic Regression (Tuned)",
"Decision Tree (Tuned)",
"Random Forest (Tuned)"
],
"Accuracy": [
lr_tuned["Accuracy"].iloc[0],
dt_tuned["Accuracy"].iloc[0],
rf_tuned["Accuracy"].iloc[0]
],
"Precision": [
lr_tuned["Precision"].iloc[0],
dt_tuned["Precision"].iloc[0],
rf_tuned["Precision"].iloc[0]
],
"Recall": [
lr_tuned["Recall"].iloc[0],
dt_tuned["Recall"].iloc[0],
rf_tuned["Recall"].iloc[0]
],
"F1_score": [
lr_tuned["F1_Score"].iloc[0],
dt_tuned["F1_Score"].iloc[0],
rf_tuned["F1_Score"].iloc[0]
],
"AUC_ROC": [
lr_tuned["AUC_ROC"].iloc[0],
dt_tuned["AUC_ROC"].iloc[0],
rf_tuned["AUC_ROC"].iloc[0]
]
})

comparison_df = comparison_df.set_index("Model")

print("Tuned models comparison:")
display(comparison_df.round(4))

# Save comparison table
output_dir = "../outputs/model_comparison"
os.makedirs(output_dir, exist_ok=True)

comparison_path = os.path.join(output_dir, "model_comparison_metrics.csv")
comparison_df.to_csv(comparison_path)

### 6.4 Plot metric comparison

In [ ]:
metrics_to_plot = ["Accuracy", "Precision", "Recall", "F1_score", "AUC_ROC"]

for metric in metrics_to_plot:
    plt.figure(figsize=(6, 4))
    
    # Bar plot for current metric
    comparison_df[metric].plot(kind="bar")
    
    # Labels and title
    plt.title(metric + " comparison between tuned models")
    plt.ylabel(metric)
    plt.xlabel("Model")
    plt.xticks(rotation=15)
    plt.tight_layout()
    
    # Save figure to outputs folder
    fig_path = os.path.join(
        "../outputs/model_comparison",
        f"model_comparison_{metric.lower()}.png"
    )
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    
    # Show plot
    plt.show()
    

### 6.5 Identify best model for each metric

In [ ]:
metrics_to_plot = ["Accuracy", "Precision", "Recall", "F1_score", "AUC_ROC"]

best_models = {}

for metric in metrics_to_plot:
    best_model_name = comparison_df[metric].idxmax()
    best_value = comparison_df.loc[best_model_name, metric]
    best_models[metric] = (best_model_name, best_value)

print(
    "\nBest Tuned Model Summary\n"
    "________________________________________________________\n"
)

for metric in metrics_to_plot:
    name = comparison_df[metric].idxmax()
    value = comparison_df[metric].max()
    print(f"{metric:<10}: {name:<30} Score: {value:.4f}")
